### Simple GenAI App using Langchain and Ollama

In [23]:
#Data Ingestion - From the website - scrape data
from langchain_community.document_loaders import WebBaseLoader

In [24]:
loader = WebBaseLoader('https://docs.langchain.com/langsmith/administration-overview')
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageLangSmith setupSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationOrganizationOverviewOverviewAccountCloudBYOCSelf-hostedGovernOverviewOrganizationOverviewWorkspace setupManage organizations using the APITerraform providerUsers & access controlUser managementRole-based access controlAttribute-based access controlOrganization and workspace operationsAuthentication methodsUser access in SSO organizationsToolsChatCLISkillsAuditingAudit logsData & compliance

In [25]:
#Data Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)
split_docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content="Overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageLangSmith setupSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationOrganizationOverviewOverviewAccountCloudBYOCSelf-hostedGovernOverviewOrganizationOverviewWorkspace setupManage organizations using the APITerraform providerUsers & access controlUser managementRole-based access controlAttribute-based access controlOrganization and workspace operationsAuthentication methodsUser access in SSO organizationsToolsChatCLISkillsAuditingAudit logsData & complianceS

In [31]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model = 'embeddinggemma')

In [32]:
#Storing the vectors into a vector database
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(split_docs,embeddings)
vectorstoredb

In [36]:
#query from vectorstoredb
query = 'An application is a logical grouping of resources'
ans = vectorstoredb.similarity_search(query)
ans

[Document(id='e9cb38c4-1351-4cf6-bda2-b38628efc8bc', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Switch applications from the main navigation sidebar in the LangSmith UI. Use the Application dropdown at the top of the sidebar to select an application.\nAny resource can be created without being tagged to an application. These resources will be visible when the All applications option is selected.\n\u200bResources\nResources are the concrete entities used to build, run, and observe applications and agents, such as tracing projects, prompts, datasets, and deployments. Resources are scoped to a specific application.\n\u200bAdditional info\nThe following diagram explains the relationship between organizations, workspaces, applications, and resources: \nSee the table below for details on which features are available in which scope(s):'),
 Document(id='0da201fa-b396-4044-b4d3-5ab9

In [37]:
ans[0].page_content

'Switch applications from the main navigation sidebar in the LangSmith UI. Use the Application dropdown at the top of the sidebar to select an application.\nAny resource can be created without being tagged to an application. These resources will be visible when the All applications option is selected.\n\u200bResources\nResources are the concrete entities used to build, run, and observe applications and agents, such as tracing projects, prompts, datasets, and deployments. Resources are scoped to a specific application.\n\u200bAdditional info\nThe following diagram explains the relationship between organizations, workspaces, applications, and resources: \nSee the table below for details on which features are available in which scope(s):'

In [38]:
from langchain_community.llms import Ollama
llm = Ollama(model='llama3')
llm

Ollama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, model='llama3')

In [39]:
#Retrieval Chain, Document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>
    """
)

document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n    '), additional_kwargs={})])
| Ollama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, model='llama3')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [43]:
from langchain_core.documents import Document 
document_chain.invoke({
    'input':'An application is a logical grouping of resources',
    'context':[Document(page_content='An application is a logical grouping of resources within a workspace. Applications are often agents, but you can use them for any project within a team. Applications keep the UI organized by only surfacing the resources associated with the application currently in context. Applications are built on top of resource tags and can be used to control resource access using ABAC.')]
})

'What is the purpose of an application in a workspace?'

We want documents to first come from a retriever we will set up so that we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [44]:
#Input --> Retriever --> Vectorstoredb
retriever = vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020FC4AED520>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n    '), additional_kwargs={})])
            | 

In [45]:
#Getting the final response using the retriever
response = retrieval_chain.invoke({'input':'An application is a logical grouping of resources'})
response['answer']

'Based on the provided context, you can switch applications from the main navigation sidebar in the LangSmith UI by using the Application dropdown at the top of the sidebar to select an application.'

In [46]:
response

{'input': 'An application is a logical grouping of resources',
 'context': [Document(id='e9cb38c4-1351-4cf6-bda2-b38628efc8bc', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Switch applications from the main navigation sidebar in the LangSmith UI. Use the Application dropdown at the top of the sidebar to select an application.\nAny resource can be created without being tagged to an application. These resources will be visible when the All applications option is selected.\n\u200bResources\nResources are the concrete entities used to build, run, and observe applications and agents, such as tracing projects, prompts, datasets, and deployments. Resources are scoped to a specific application.\n\u200bAdditional info\nThe following diagram explains the relationship between organizations, workspaces, applications, and resources: \nSee the table below for details on which features are

In [47]:
response['context']

[Document(id='e9cb38c4-1351-4cf6-bda2-b38628efc8bc', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Switch applications from the main navigation sidebar in the LangSmith UI. Use the Application dropdown at the top of the sidebar to select an application.\nAny resource can be created without being tagged to an application. These resources will be visible when the All applications option is selected.\n\u200bResources\nResources are the concrete entities used to build, run, and observe applications and agents, such as tracing projects, prompts, datasets, and deployments. Resources are scoped to a specific application.\n\u200bAdditional info\nThe following diagram explains the relationship between organizations, workspaces, applications, and resources: \nSee the table below for details on which features are available in which scope(s):'),
 Document(id='0da201fa-b396-4044-b4d3-5ab9